<div style="background: linear-gradient(135deg, #050510 0%, #0f0f2d 35%, #1a0a2e 70%, #0a1a1a 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #7b2fff44;"><h1 style="color: #a855f7; font-family: 'Courier New', monospace; font-size: 2.9em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #a855f766;">🎥 OpenCV DEVAM</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.35em; margin: 0 0 18px 0; font-weight: 300;">6. Hafta — Kamera, Video Kaydetme, Piksel İşlemleri & Fare Olayları</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">VideoCapture &bull; VideoWriter &bull; Renk Kanalları &bull; Mouse Callback &bull; ROI &bull; Görüntü Kırpma</p></div>

---
# 📡 BÖLÜM 1: Kamera ve Video Mimarisi

## 1.1 Dijital Kameranın Anatomisi

Kamera görüntüsünün bilgisayara gelene kadar geçtiği yol:

```
Işık
  ↓
Optik Lens (odaklama)
  ↓
Görüntü Sensörü [CMOS / CCD]
(Her piksel bir fotodiod — foton → elektron → voltaj)
  ↓
ADC — Analog Dijital Dönüştürücü
(Voltaj → 8/12/14 bit sayı)
  ↓
ISP — Image Signal Processor
(Gürültü azaltma, renk düzeltme, HDR, beyaz dengesi)
  ↓
Video Codec (H.264/H.265 sıkıştırma)
  ↓
USB / PCIe / MIPI arayüzü
  ↓
OpenCV — VideoCapture
  ↓
Uygulamanız
```

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>CMOS vs CCD — Sensör Savaşı:</b><br><br>Dijital kameralarda iki ana sensör teknolojisi kullanılır. <b>CCD (Charge-Coupled Device)</b> 1969'da Bell Laboratories'de Willard Boyle ve George Smith tarafından icat edildi (2009 Nobel Fizik Ödülü). Yüksek kalite, düşük gürültü — ama pahalı ve yavaş.<br><br><b>CMOS (Complementary Metal-Oxide-Semiconductor)</b> her pikselin kendi amplifikatörüne sahip olduğu yapı. 1990'larda enerji verimliliği ve maliyet avantajı ile yaygınlaştı. Bugün akıllı telefonlar, webcam'ler ve profesyonel DSLR'ların neredeyse tamamı CMOS kullanır.<br><br>Önemli nokta: iPhone'unuzdaki kamera sensörü, 1969'da bilgisayar odası büyüklüğündeki sistemlerden daha iyi görüntü üretiyor.</span></div>

## 1.2 VideoCapture — ID Sistemi

```python
cv.VideoCapture(0)   # Varsayılan webcam (dahili)
cv.VideoCapture(1)   # İkinci kamera (harici USB)
cv.VideoCapture(2)   # Üçüncü kamera
cv.VideoCapture("video/ornek.mp4")  # Video dosyası
cv.VideoCapture("rtsp://192.168.1.100:554/stream")  # IP kamera
cv.VideoCapture("https://example.com/stream.m3u8")  # HLS stream
```

`isOpened()` mutlaka kontrol edilmeli — kamera meşgul, izin yok veya takılı değilse `False` döner ve `read()` çağrısı program çökmesine yol açar.

---
# 📷 BÖLÜM 2: Kameradan Görüntü Alma

## 2.1 Temel Kamera Döngüsü

OpenCV kamera kullanımının temel şablonu — her projede bu yapı tekrar eder:

In [ ]:
import cv2 as cv

# Kamera aç — 0: varsayılan dahili kamera, 1: USB kamera
video = cv.VideoCapture(1)

# Kamera özelliklerini oku
fps = video.get(cv.CAP_PROP_FPS)
en  = video.get(cv.CAP_PROP_FRAME_WIDTH)
boy = video.get(cv.CAP_PROP_FRAME_HEIGHT)
print(f"Kamera: {int(en)}x{int(boy)} @ {fps:.1f} FPS")

while video.isOpened():
    ret, frame = video.read()  # ret: başarı mı?, frame: kare

    if ret:   # kare başarıyla okunduysa
        cv.imshow("kamera", frame)

        if cv.waitKey(1) == ord("q"):  # q'ya basınca çık
            break

# Temizlik — ZORUNLU!
video.release()          # kamera kaynağını serbest bırak
cv.destroyAllWindows()   # tüm pencereleri kapat

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>video.release() neden zorunlu?</b><br><br>Kamera, işletim sisteminde bir <b>exclusive kaynak</b>tır: aynı anda yalnızca bir program kullanabilir. Programınız çöker veya release() çağrılmazsa kamera "meşgul" kalır — bir sonraki çalıştırmada <code>VideoCapture(0)</code> başarısız olabilir.<br><br><code>try/finally</code> bloğu kullanmak daha güvenlidir:<br><code>try:<br>&nbsp;&nbsp;# kamera kodu<br>finally:<br>&nbsp;&nbsp;video.release()<br>&nbsp;&nbsp;cv.destroyAllWindows()</code></span></div>

## 2.2 Kamera Görüntüsünü Gri Tonlamaya Çevirme

Renk uzayı dönüşümü ile kamera akışını işleyerek gösterme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)

while video.isOpened():
    ret, frame = video.read()

    # BGR → Gri dönüşümü
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    if ret:
        cv.imshow("kamera", gray)
        if cv.waitKey(1) == ord("q"):
            break

video.release()
cv.destroyAllWindows()

## 2.3 Nişangah (Crosshair) Ekleme

Görüntü üzerine piksel manipülasyonuyla nişangah çizme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)

while video.isOpened():
    ret, frame = video.read()
    nisangah = frame.copy()   # orijinali bozma

    if ret:
        # Yatay çizgi: tüm satır 240 = 0 (siyah)
        nisangah[239, :] = 0
        nisangah[240, :] = 0
        nisangah[241, :] = 0

        # Dikey çizgi: tüm sütunlar 319-321 = 0
        nisangah[:, 319:322] = 0

        cv.imshow("kamera", nisangah)
        if cv.waitKey(33) == 27:   # ESC = 27
            break

video.release()
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>frame.copy() vs doğrudan değiştirme:</b><br><br>NumPy'da <code>nisangah = frame</code> yazmak bir referans kopyası oluşturur — ikisi aynı bellek adresine işaret eder. <code>nisangah[240, :] = 0</code> yazdığınızda <b>hem nisangah hem de frame</b> değişir.<br><br><code>nisangah = frame.copy()</code> ise bağımsız bir bellek bloğu oluşturur. Orijinal kare değişmez — bir sonraki işleme adımına temiz veri gider.<br><br>Video döngüsünde bu fark kritiktir: aynı kareyi birden fazla farklı işlemle (gri, kenarlı, renklendirilmiş) ayrı pencerelerde göstermek istiyorsanız her işlem için <code>frame.copy()</code> kullanmalısınız.</span></div>

---
# 🌈 BÖLÜM 3: Renk Kanalı Manipülasyonu

## 3.1 Kırmızı ve Yeşil Kanalları Değiştirme

Kamera akışında renk kanallarını birbirleriyle değiştirme (swap) ve yarıya indirme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)

while video.isOpened():
    ret, frame = video.read()

    if not ret:
        break

    # ── Kanal kopyaları al (önemli: önce kopyala, sonra değiştir)
    kirmizi = frame[:, :, 2].copy()   # Kırmızı kanal (BGR'de indeks 2)
    yesil   = frame[:, :, 1].copy()   # Yeşil kanal

    # Kanalları çapraz atama: kırmızı↔yeşil
    frame[:, :, 2] = yesil    # Kırmızı → Yeşil
    frame[:, :, 1] = kirmizi  # Yeşil → Kırmızı

    # Mavi kanalı yarıya düşür
    frame[:, :, 0] = frame[:, :, 0] // 2

    cv.imshow("kamera", frame)
    if cv.waitKey(1) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

<div style="background: #2d2200; border-left: 5px solid #f9c74f; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f9c74f; font-size: 1.08em;">⚠️ Kritik Nokta</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Kanal değiştirmede sıra önemli!</b><br><br><code>frame[:,:,2] = frame[:,:,1]</code> yazarsanız önce kırmızıyı yeşille ezersiniz, sonra <code>frame[:,:,1] = frame[:,:,2]</code> yazdığınızda artık kırmızı değil <b>yeşil kanalı</b> okursunuz — ikisi de yeşile eşitlenir!<br><br>Doğru yol: <b>önce bir kanalı geçici değişkene al</b> (<code>.copy()</code>), sonra değiştir. Bu, klasik "iki değişkeni değiştirme" problemiyle aynı mantıktır: <code>a, b = b, a</code> Python'da doğal çalışır ama NumPy dilimlemesinde geçici kopya şarttır.</span></div>

## 3.2 Renk Kanallarını Ayrı Pencerelerde Görüntüleme

Görüntünün BGR kanallarını ayrı ayrı inceleme:

In [ ]:
import cv2 as cv

# Statik görüntü üzerinde kanal analizi
resim = cv.imread("resim/kirpik_bugday.jpg")

print("Görüntü boyutu:", resim.shape)

# Kanal analizi
print("\nKanal İstatistikleri:")
print(f"  Mavi  ortalama: {resim[:,:,0].mean():.1f}")
print(f"  Yeşil ortalama: {resim[:,:,1].mean():.1f}")
print(f"  Kırmızı ortalama: {resim[:,:,2].mean():.1f}")

# Her kanalı ayrı pencerede göster
cv.imshow("mavi",    resim[:, :, 0])   # yalnızca mavi
cv.imshow("yesil",   resim[:, :, 1])   # yalnızca yeşil
cv.imshow("kirmizi", resim[:, :, 2])   # yalnızca kırmızı
cv.waitKey(0)
cv.destroyAllWindows()

# Açıklama: kanal görüntüleri gri tonlamalı görünür
# Çünkü her kanal tek boyutlu (2D) — renk yok
print("\nNot: ayrı kanallar gri görünür çünkü 2D (tek kanal) — renk bilgisi yok.")

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Neden Görüntüler Gri Görünür? — Tek Kanal Mantığı</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Bir görüntü kanalını tek başına gösterdiğinizde (<code>resim[:,:,0]</code>) shape=(H,W) olan 2D bir dizi elde edersiniz. OpenCV bu diziyi gri tonlamalı olarak yorumlar: 0=siyah, 255=beyaz, arası gri tonlar.<br><br><b>Renkli kanal görüntüsü için:</b> Yalnızca bir kanalı doldurun, diğerlerini sıfır yapın:<br><code>sadece_mavi = np.zeros_like(resim)<br>sadece_mavi[:,:,0] = resim[:,:,0]  # yalnızca mavi kanal dolu</code><br><br>Bu teknik, görüntü analitiklerinde "hangi renk bölgede hâkim?" sorusunu yanıtlamak için kullanılır. Uydu görüntülerinde bitki örtüsünü tespit etmek için kırmızı kanalın yeşil kanaldan çıkarılması (NDVI) tam bu mantıkla çalışır.</span></div>

---
# 📼 BÖLÜM 4: Video Kaydetme — `cv.VideoWriter`

## 4.1 Video Codec'leri — Derinlemesine

```python
fourcc = cv.VideoWriter_fourcc(*"XVID")  # FourCC kodu
```

**FourCC (Four Character Code):** Video codec'ini 4 ASCII karakterle tanımlayan sistem.

| FourCC | Codec | Format | Özellik |
|--------|-------|--------|--------|
| `XVID` | MPEG-4 | .avi | Geniş uyumluluk, OpenCV varsayılanı |
| `MJPG` | Motion JPEG | .avi | Kare başına JPEG, yüksek kalite |
| `mp4v` | MPEG-4 | .mp4 | Modern, taşınabilir |
| `avc1` | H.264 | .mp4 | En yaygın web formatı |
| `H265` | H.265/HEVC | .mp4 | 4K için verimli |
| `0` | Ham | .avi | Sıkıştırmasız (dev dosya) |

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Video Codec Tarihinin Dönüm Noktaları:</b><br><br><b>DivX (2000):</b> MPEG-4 tabanlı codec. İnternet üzerinden film paylaşımını mümkün kıldı — hem yasal hem de "diğer yollarla". Film endüstrisini sarstı, dijital dağıtım tartışmalarını başlattı.<br><br><b>XVID (2001):</b> DivX'in açık kaynak kardeşi. OpenCV'nin hâlâ varsayılan olarak kullandığı codec.<br><br><b>H.264 (2003):</b> Modern internetin video dili. YouTube, Netflix, Zoom, Teams — hepsi H.264. Blu-ray'in de temel codec'i. İnsan hayatına en çok dokunan algoritmalardan biri olduğu söylenebilir.<br><br><b>VP9 ve AV1:</b> Google ve Alliance for Open Media'nın patent-free alternatifleri. YouTube ve Chrome AV1'e geçiş yapıyor.</span></div>

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)

# Video yazıcı ayarları
fourcc = cv.VideoWriter_fourcc(*"XVID")   # codec seçimi
out = cv.VideoWriter(
    "video/kayit.avi",  # çıktı dosyası
    fourcc,             # codec
    20.0,               # FPS
    (640, 480),         # çözünürlük (genişlik, yükseklik)
    0                   # 0=gri, 1=renkli (varsayılan)
)

print("Kayıt başlıyor... q ile dur")

while video.isOpened():
    ret, frame = video.read()
    nisangah = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    if ret:
        # Nişangah çiz
        nisangah[239, :] = 0
        nisangah[240, :] = 0
        nisangah[241, :] = 0
        nisangah[:, 319:322] = 0

        out.write(nisangah)          # kareyi dosyaya yaz
        cv.imshow("kamera", nisangah)

        if cv.waitKey(33) == ord("q"):
            break

out.release()     # video dosyasını kapat (zorunlu!)
video.release()
cv.destroyAllWindows()
print("Kayıt tamamlandı: video/kayit.avi")

## 4.2 Reklam Çerçevesi Ekleme — Periyodik Efekt

Her 10 karede bir reklamı flaş gibi gösterme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)
fourcc = cv.VideoWriter_fourcc(*"XVID")
out = cv.VideoWriter("video/kayit.avi", fourcc, 20.0, (640, 480))

resim = cv.imread("resim/ad_soyad.jpg")

i = 0
while video.isOpened():
    i += 1
    ret, frame = video.read()

    # Her 10 karede 1 kare reklam göster
    if i % 10 == 0:
        frame = resim   # reklam görüntüsü
        i = 0

    if ret:
        out.write(frame)
        cv.imshow("kamera", frame)
        if cv.waitKey(33) == ord("q"):
            break

out.release()
video.release()
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Broadcast ve Streaming Sistemlerinde Periyodik İçerik:</b><br><br>Bu basit <code>i % 10 == 0</code> mantığı, yayıncılık sistemlerinin temelindeki fikrin küçük ölçekli modelidir. TV yayınlarında reklam arası saatlere göre programlanır. Canlı spor yayınlarında skor tabelası belirli aralıklarla yenilenir. Güvenlik kamera sistemlerinde hareket algılandığında kayıt başlar, olmadığında yalnızca düşük çözünürlüklü kare atlanır (frame skipping). Hepsi aynı "koşula göre akış kontrolü" mantığını kullanır.</span></div>

---
# 📐 BÖLÜM 5: Görüntü Boyutu ve Çözünürlük

## 5.1 Çözünürlük Kavramı — Derinlemesine

Görüntü boyutu çeşitli şekillerde ifade edilir:

```
resim.shape = (720, 1280, 3)
              ↑     ↑     ↑
           satır  sütun  kanal
          yüksek  geniş

Dikkat: "720p" → 1280×720 (genişlik×yükseklik)
Ama shape: (720, 1280, 3) (yükseklik, genişlik, kanal)

Resolution terminolojisi:
  720p  = 1280×720  HD
  1080p = 1920×1080 Full HD
  1440p = 2560×1440 QHD (2K)
  2160p = 3840×2160 4K / UHD
  4320p = 7680×4320 8K
```

In [ ]:
import cv2 as cv

resim = cv.imread("resim/ad_soyad.jpg")

print("=== Görüntü Bilgileri ===")
print(f"Shape:    {resim.shape}")

yukseklik, genislik, kanallar = resim.shape
print(f"Genişlik: {genislik} piksel")
print(f"Yükseklik:{yukseklik} piksel")
print(f"Kanallar: {kanallar} (BGR)")
print(f"Toplam:   {genislik*yukseklik:,} piksel")
print(f"RAM:      {resim.nbytes/1024:.1f} KB")

# İlk pikselin BGR değeri
print(f"\n[0,0] piksel BGR: {resim[0,0]}")
print(f"[0,0] piksel B={resim[0,0,0]}, G={resim[0,0,1]}, R={resim[0,0,2]}")

---
# 🖊️ BÖLÜM 6: Piksel Okuma ve Değiştirme

## 6.1 Tek Piksel İşlemleri

Bireysel piksel değerlerini okuma ve değiştirme:

In [ ]:
import cv2 as cv

resim = cv.imread("resim/ad_soyad.jpg")

# Piksel okuma
print("İlk piksel [0,0] değeri:", resim[0, 0])
cv.imshow("a", resim)
cv.waitKey(0)

# Tek piksel değiştir
resim[10, 10] = [0, 0, 0]   # siyah nokta (gözle neredeyse görünmez)
print("Değiştirilen piksel:", resim[10, 10])

cv.imshow("a", resim)
cv.waitKey(0)

cv.imwrite("resim/ad_soyad_benekli.jpg", resim)

# Bölge değiştir
resim[10:50, 10:50] = [0, 0, 0]   # 40×40 siyah kare

cv.imshow("a", resim)
cv.waitKey(0)

cv.imwrite("resim/ad_soyad_benekli_2.jpg", resim)
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Piksel Manipülasyonu Neden Yavaş? — Python'un Bedeli:</b><br><br>Tek tek piksel değiştirmek (<code>for i in range(H): for j in range(W): resim[i,j]=...</code>) Python'da son derece yavaştır. 1080p görüntü = 2 milyon piksel — Python döngüsü ile bu işlem <b>10+ saniye</b> sürer.<br><br>OpenCV'nin çözümü: vektörize işlemler kullanmak. <code>resim[10:50, 10:50] = 0</code> tek satır — C++ seviyesinde çalışır, aynı işlemi mikrosaniyeler içinde yapar.<br><br><b>Gerçek dünya kuralı:</b> "Eğer piksel piksel döngü yazıyorsanız, büyük ihtimalle daha iyi bir NumPy/OpenCV yolu vardır."</span></div>

## 6.2 Kırmızı Pikselleri Yok Etme — Kanal Sıfırlama

Görüntüdeki kırmızı kanalını tamamen sıfırlama, iki farklı yöntem:

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/manzara.jpg")
print(f"Orijinal şekil: {resim.shape}")
print(f"[0,0] piksel: {resim[0, 0]}")

# YÖNTEM 1: for döngüsü (yavaş ama anlaşılır)
# for a in resim:
#     for b in a:
#         b[2] = 0   # Kırmızı kanalı (BGR indeks 2) sıfırla

# YÖNTEM 2: NumPy dilimleme (binlerce kat daha hızlı!)
resim[:, :, 2] = 0   # Tüm satırlar, tüm sütunlar, 2. kanal (R) = 0

print(f"Kırmızısız [0,0] piksel: {resim[0, 0]}")
cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

---
# ✂️ BÖLÜM 7: Görüntü Kırpma (Cropping)

## 7.1 ROI — Region of Interest

**ROI (İlgi Alanı):** Görüntünün yalnızca belirli bir bölgesi üzerinde işlem yapmak için NumPy dilimleme kullanılır.

```
resim[y1:y2, x1:x2]
       ↑        ↑
  satır aralığı  sütun aralığı
  (yükseklik)    (genişlik)

Dikkat! shape=(H,W) → resim[satır, sütun] = resim[y, x]
Ama (x,y) koordinat sistemi tam tersi!
```

In [ ]:
import cv2 as cv

resim = cv.imread("resim/manzara.jpg")
cv.imshow("kirpilmamis", resim)
cv.waitKey(0)

print("Orijinal:", resim.shape)

# Kırpma: satır 600-900, sütun 500-1200
yol = resim[600:900, 500:1200]
print("Kırpılmış bölge:", yol.shape)
cv.imshow("a", yol)
cv.waitKey(0)
cv.imwrite("resim/kirpilmis_manzara.jpg", yol)

# İkinci bölge
cali = resim[500:750, 200:430]
print("İkinci kırpma:", cali.shape)
cv.imshow("a", cali)
cv.waitKey(0)

cv.destroyAllWindows()

## 7.2 Fare ile İnteraktif Kırpma — Mouse Callback

Kullanıcının fare ile seçtiği bölgeyi kırpma:

In [ ]:
import cv2 as cv
import time

# Global değişkenler — callback fonksiyonu erişmesi için
bas_x = bas_y = 0

def fare_tiklama_olayi(event, x, y, flags, param):
    """Mouse callback fonksiyonu — fare olaylarını yakalar."""
    global bas_x, bas_y

    if event == cv.EVENT_LBUTTONDOWN:   # Sol tuş basıldı
        bas_x = x
        bas_y = y
        print(f"Başlangıç: ({x}, {y})")

    if event == cv.EVENT_LBUTTONUP:     # Sol tuş bırakıldı
        print(f"Bitiş: ({x}, {y})")

        # min/max: hangi yönde sürüklediğinden bağımsız
        kirpilmis = resim[
            min(bas_y, y) : max(bas_y, y),
            min(bas_x, x) : max(bas_x, x)
        ]

        if kirpilmis.size > 0:   # boş olmadığından emin ol
            cv.imwrite("resim/kirpilmis_manzara_fare_ile.jpg", kirpilmis)
            cv.imshow("Kirpilmis Resim", kirpilmis)
            print(f"Kırpıldı: {kirpilmis.shape} → kaydedildi")


resim = cv.imread("resim/manzara.jpg")
cv.imshow("Fare", resim)
cv.setMouseCallback("Fare", fare_tiklama_olayi)  # callback bağla
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Mouse Callback — Event-Driven Programlama</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Event-driven (olay güdümlü) programlama</b> modern GUI'ların temelidir. Program "bekler", kullanıcı bir şey yapınca işler. OpenCV'nin mouse callback sistemi bu paradigmanın basit bir örneğidir.<br><br>Daha zengin bir event sistemi için <code>flags</code> parametresi kullanılabilir:<br><code>cv.EVENT_FLAG_CTRLKEY</code> → Ctrl basılı mı?<br><code>cv.EVENT_FLAG_SHIFTKEY</code> → Shift basılı mı?<br><code>cv.EVENT_FLAG_LBUTTON</code> → Sol tuş basılı tutulurken hareket<br><br>Bir grafik editörü yapıyorsanız: seçim aracı LBUTTONDOWN + MOUSEMOVE + LBUTTONUP kombinasyonunu takip eder — tam bu hafta öğrendiğiniz yapı!</span></div>

---
# 🖱️ BÖLÜM 8: Gelişmiş Fare ile Bölge Seçme Uygulaması

## 8.1 Dondurulmuş Kare + ROI Seçimi

`s` ile görüntüyü dondur, fare ile bölge seç, `d` ile devam et — gerçekçi bir interaktif uygulama mimarisi:

In [ ]:
import cv2 as cv
import os

# ── Durum değişkenleri ──────────────────────────────────────────────────
secim_basladi   = False
baslangic_nokta = (-1, -1)
bitis_nokta     = (-1, -1)
secim_yapiliyor = False
dondurulmus_frame = None
bolge_sayaci    = 1

kayit_klasoru = "resim"
os.makedirs(kayit_klasoru, exist_ok=True)

# ── Mouse callback ──────────────────────────────────────────────────────
def mouse_callback(event, x, y, flags, param):
    global secim_basladi, baslangic_nokta, bitis_nokta
    global secim_yapiliyor, bolge_sayaci, dondurulmus_frame

    if event == cv.EVENT_LBUTTONDOWN:
        secim_basladi   = True
        secim_yapiliyor = True
        baslangic_nokta = (x, y)
        bitis_nokta     = (x, y)

    elif event == cv.EVENT_MOUSEMOVE and secim_yapiliyor:
        bitis_nokta = (x, y)   # canlı önizleme için koordinat güncelle

    elif event == cv.EVENT_LBUTTONUP:
        secim_yapiliyor = False
        bitis_nokta     = (x, y)

        x1 = min(baslangic_nokta[0], bitis_nokta[0])
        y1 = min(baslangic_nokta[1], bitis_nokta[1])
        x2 = max(baslangic_nokta[0], bitis_nokta[0])
        y2 = max(baslangic_nokta[1], bitis_nokta[1])

        if (x2 - x1) > 5 and (y2 - y1) > 5 and dondurulmus_frame is not None:
            bolge = dondurulmus_frame[y1:y2, x1:x2]
            dosya_adi = os.path.join(kayit_klasoru, f"bolge_{bolge_sayaci}.png")
            cv.imwrite(dosya_adi, bolge)
            print(f"[✓] Bölge {bolge_sayaci} → {dosya_adi}  boyut:{bolge.shape}")
            bolge_sayaci += 1

# ── Ana döngü ────────────────────────────────────────────────────────────
video = cv.VideoCapture(0)
cv.namedWindow("kamera")
cv.setMouseCallback("kamera", mouse_callback)

donduruldu = False
print("Kontroller: [S]=dondur  [D]=devam  [ESC]=çık")

while video.isOpened():
    if not donduruldu:
        ret, frame = video.read()
        if not ret:
            break

        # Renk kanalı manipülasyonu
        kirmizi = frame[:, :, 2].copy()
        yesil   = frame[:, :, 1].copy()
        frame[:, :, 2] = yesil
        frame[:, :, 1] = kirmizi
        frame[:, :, 0] = frame[:, :, 0] // 2

        dondurulmus_frame = frame.copy()
        gosterim = frame.copy()
    else:
        gosterim = dondurulmus_frame.copy()

        # Seçim dikdörtgenini gerçek zamanlı çiz
        if baslangic_nokta != (-1, -1):
            x1 = min(baslangic_nokta[0], bitis_nokta[0])
            y1 = min(baslangic_nokta[1], bitis_nokta[1])
            x2 = max(baslangic_nokta[0], bitis_nokta[0])
            y2 = max(baslangic_nokta[1], bitis_nokta[1])
            cv.rectangle(gosterim, (x1, y1), (x2, y2), (0, 255, 255), 2)

        cv.putText(gosterim, "DONDURULDU | Bolge sec | [D] devam",
                   (10, 25), cv.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)

    cv.imshow("kamera", gosterim)
    tus = cv.waitKey(2) & 0xFF

    if tus == ord("s") and not donduruldu:
        donduruldu = True
        baslangic_nokta = bitis_nokta = (-1, -1)
        print("[S] Donduruldu — bölge seçebilirsiniz")

    elif tus == ord("d") and donduruldu:
        donduruldu = False
        baslangic_nokta = bitis_nokta = (-1, -1)
        print("[D] Devam ediyor")

    elif tus == 27:   # ESC
        break

video.release()
cv.destroyAllWindows()

## 8.2 Uygulama Mimarisi — State Machine Yaklaşımı

Yukarıdaki program bir **sonlu durum makinesi (Finite State Machine)** örneğidir:

```
          [S tuşu]
  CANLI ─────────────→ DONDURULMUŞ
    ↑                       │
    │       [D tuşu]        │
    └───────────────────────┘
    │
    │ [ESC]  → ÇIKIŞ
```

Her durumda `mouse_callback` ve ana döngü farklı davranır. Bu, GUI uygulamaları, oyun geliştirme ve robotik kontrol sistemlerinin temel tasarım desenidir.

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>State Machine Gerçek Dünya Uygulamaları:</b><br><br><b>Trafik ışığı:</b> Kırmızı → Yeşil → Sarı → Kırmızı. Her durum belirli süre kalır, belirli koşulda geçiş yapar.<br><br><b>Oyun karakteri:</b> Durma → Yürüme → Koşma → Zıplama → Düşme. Unity ve Unreal Engine'de "Animator State Machine" tam bu yapıdır.<br><br><b>TCP protokolü:</b> CLOSED → SYN_SENT → ESTABLISHED → FIN_WAIT → TIME_WAIT → CLOSED. İnternetin temel protokolü bir state machine üzerine inşa edilmiştir.<br><br><b>Robotik:</b> Arıyor → Nesne Buldu → Yaklaşıyor → Tutuyor → Taşıyor. Boston Dynamics robotlarının davranış katmanı state machine tabanlıdır.</span></div>

---
# 🎨 BÖLÜM 9: Renk Uzayı Dönüşümleri

## 9.1 Desteklenen Renk Uzayları

OpenCV 150+ renk uzayı dönüşümü destekler. En yaygın kullanılanlar:

| Kod | Dönüşüm | Kullanım |
|-----|---------|----------|
| `COLOR_BGR2GRAY` | Renkli → Gri | Kenar tespiti, eşikleme |
| `COLOR_BGR2HSV` | BGR → HSV | Renk bazlı nesne tespiti |
| `COLOR_BGR2RGB` | BGR → RGB | matplotlib, PIL uyumu |
| `COLOR_BGR2LAB` | BGR → L*a*b* | Renk farkı ölçümü |
| `COLOR_BGR2YCrCb`| BGR → YCrCb | Video sıkıştırma |
| `COLOR_BGRA2GRAY`| BGRA → Gri | Alpha kanallı görüntü |
| `COLOR_BGR2BGRA` | BGR → BGRA | Alpha kanalı ekle |

In [ ]:
import cv2 as cv
import numpy as np

# 4 kanallı (BGRA) görüntü okuma ve dönüşüm
resim = cv.imread("resim/b.tif", cv.IMREAD_UNCHANGED)
print("Orijinal shape:", resim.shape)

gri = cv.cvtColor(resim, cv.COLOR_BGRA2GRAY)
print("Gri shape:", gri.shape)

cv.imshow("a", gri)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

# BGR → BGRA (alpha kanalı ekleme)
resim = cv.imread("resim/a.png", cv.IMREAD_UNCHANGED)
print("Orijinal:", resim.shape)

seffaf = cv.cvtColor(resim, cv.COLOR_BGR2BGRA)
print("BGRA:", seffaf.shape)

cv.imshow("a", seffaf)
cv.waitKey(0)
cv.imwrite("resim/a_alpha_kanalli.png", seffaf)
cv.destroyAllWindows()

# Not: eklenen alpha kanalı başlangıçta 255 (tamamen opak)
print(f"Alpha kanalı örnek değerleri: {seffaf[0:3, 0:3, 3]}")

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 L*a*b* Renk Uzayı — İnsanın Algısına En Yakın</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>CIE L*a*b* (1976):</b> Uluslararası Aydınlatma Komisyonu tarafından tanımlanan, insan görsel algısına en yakın renk uzayıdır.<br><br><b>L*:</b> Parlaklık (0=siyah, 100=beyaz)<br><b>a*:</b> Yeşil (-) ile Kırmızı (+) ekseni<br><b>b*:</b> Mavi (-) ile Sarı (+) ekseni<br><br><b>Önemli özellik: Öklid mesafesi = Görsel fark!</b><br>İki rengin L*a*b* uzayındaki Öklid mesafesi (ΔE), insan gözünün hissettiği farka doğrusal olarak karşılık gelir. RGB'de bu mümkün değildir.<br><br>Kullanım alanları: baskı kalitesi kontrolü, renk uyumu, görüntü karşılaştırma algoritmaları, renk körlüğü simülasyonu. Pantone ve profesyonel baskı endüstrisi standart renk farkı ölçümünde ΔE kullanır.</span></div>

---
# 🔧 BÖLÜM 10: Kapsamlı Örnek — Zaman Bükme Filtresi

Kameranın alt yarısını geciktirerek "zaman bükme" efekti:

In [ ]:
import cv2 as cv
import numpy as np

video = cv.VideoCapture(0)

# İlk kareyi oku — gecikmeli tampon başlatmak için
ret, frame = video.read()
asagi_filtreli_taraf = frame[240:, :].copy()  # alt yarı tamponu

print("Zaman bükme filtresi başlıyor — q ile çık")
print("Ekranın alt yarısı geciktirilmiş görüntü gösterecek")

while True:
    ret, frame = video.read()
    asagi = frame[240:, :]   # güncel alt yarı

    # Tamponu aşağı kaydır: her satır bir sonraki eski satır olsun
    for s in range(239, 0, -1):
        asagi_filtreli_taraf[s] = asagi_filtreli_taraf[s - 1]
    asagi_filtreli_taraf[0, :] = frame[240, :]   # sadece ilk satır güncellenir

    # Gecikmiş alt yarıyı kareye yerleştir
    frame[240:, :] = asagi_filtreli_taraf

    cv.imshow("Zaman Bükme Filtresi", frame)

    if cv.waitKey(5) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Zaman Bükme Filtresi — Teknik Analiz:</b><br><br>Bu algoritma bir <b>FIFO kuyruğu (First In First Out)</b> uygulamasıdır. Her yeni karedeki alt yarının yalnızca en üst satırı tampona eklenir, geri kalanlar bir satır aşağı kayar. Sonuç: alt yarı görüntüsü gerçek zamandan giderek uzaklaşan bir yankı (echo) oluşturur.<br><br>Bu teknik, TV yayıncılığındaki <b>frame buffer</b> ve <b>delay line</b> kavramlarının minik bir uygulamasıdır. Ses işlemede yankı efekti (reverb) da benzer bir ring buffer mantığıyla çalışır.<br><br><b>Performans notu:</b> <code>for s in range(239, 0, -1)</code> döngüsü yavaştır. Profesyonel uygulama için <code>np.roll(asagi_filtreli_taraf, 1, axis=0)</code> kullanılabilir — aynı işlevi tek satırda NumPy ile yapar.</span></div>

---
# 🏛️ BÖLÜM 11: Üniversite Seviyesi Alıştırmalar

Aşağıdaki sorular bu haftanın konularını ileri düzey bilgisayar bilimi ve görüntü işleme kavramlarıyla birleştirmenizi gerektirir. **Cevaplar verilmemiştir — çözüm sürecini düşünmek öğrenmenin kendisidir.**

---
## ❓ Soru 1 — Gerçek Zamanlı Renk Paletinden Piksel Seçici

**Zorluk: ⭐⭐⭐☆☆**

Kameradan gelen görüntüde kullanıcının fare ile tıkladığı pikselin rengini hem sayısal değerler hem görsel renk kutucuğu olarak anlık gösteren bir araç yazın. Renk uzayı karşılaştırması da yapılmalıdır.

### Görev:
1. Kameradan sürekli kare oku
2. Mouse callback ile tıklanan pikselin koordinatını kaydet
3. O pikselin rengini şu uzaylarda göster: **BGR, HSV, L\*a\*b\*, HEX**
4. Ekranın sağına bilgi paneli oluştur (`np.hstack`):
   - 50×50 piksellik renk önizleme kutusu
   - BGR, HSV, L\*a\*b\*, HEX değerleri `cv.putText` ile
   - Son 5 seçilen rengin renk şeridi
5. `c` tuşuna basınca renk geçmişini temizle

### Düşünmeniz Gereken Sorular:
- Aynı fiziksel rengi farklı uzaylarda ifade etmenin matematiksel dönüşüm zinciri nedir?
- HEX kod neden grafik tasarımda yaygın ama görüntü işlemede kullanılmaz?
- Gerçek kamera piksellerinde gürültü var; tıklanan pikselin rengi çevresindeki piksellerin ortalamasını almak neden daha güvenilir olur?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

secilen_renkler = []   # son 5 rengin geçmişi

def piksel_rengi_goster(event, x, y, flags, param):
    # tıklanan pikselin renk uzayı değerlerini hesapla
    pass

video = cv.VideoCapture(0)
# TODO: döngü ve panel
```

---
## ❓ Soru 2 — Hareket Tespiti ve Alarm Sistemi

**Zorluk: ⭐⭐⭐⭐☆**

Kamera görüntüsündeki hareketi algılayan, hareket miktarını ölçen ve belirli eşiği aştığında "alarm" veren bir güvenlik kamerası simülasyonu yazın.

### Görev:
1. Arka plan modelini öğren: ilk N kareyi ortalayarak arka plan oluştur
2. Her yeni kareyi arka planla karşılaştır: `fark = |kare - arkaplan|`
3. Fark görüntüsünü eşikleme: `cv.threshold(fark, 30, 255, THRESH_BINARY)`
4. Beyaz piksel sayısını ölç: hareket miktarı (%)
5. Eşik aşılınca: kareye kırmızı çerçeve, "HAREKET ALGILANDI" yazısı ekle
6. Hareket eden bölgenin bounding box'ını `cv.findContours` ile bul ve çiz
7. Ekrana hareket grafiği (son 100 kare) çiz

### Düşünmeniz Gereken Sorular:
- Statik arka plan yerine adaptif arka plan modellemesi (`cv.createBackgroundSubtractorMOG2`) neden daha sağlam sonuç verir?
- Hareket tespitinde ışık değişimleri (gün batımı, bulut) yanlış alarm üretir. Bu problemi nasıl azaltabilirsiniz?
- Güvenlik kameralarında **privay-preserving** (mahremiyet koruyan) hareket tespiti nasıl yapılabilir? (İnsan silueti yerine hareket ısı haritası gibi)

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np
from collections import deque

hareket_gecmisi = deque(maxlen=100)   # son 100 kare hareket miktarı

video = cv.VideoCapture(0)
# 1. Adım: ilk 30 kareyi ortalayarak arka plan oluştur
# 2. Adım: fark-eşikleme-kontur döngüsü
```

---
## ❓ Soru 3 — Sanal Beyaz Tahta (Virtual Whiteboard)

**Zorluk: ⭐⭐⭐⭐☆**

Kamerada tespit edilen renkli bir nesne (örn. kırmızı kalem ucu) sanal bir tuval üzerinde çizgi çizen bir uygulama yazın.

### Görev:
1. HSV uzayında renk maskesi ile kırmızı nesneyi tespit et
2. Nesnenin merkezini `cv.moments()` ile hesapla
3. Merkez pozisyonunu bir önceki kareden takip et — arası çizgi çiz
4. Çizilen yolları sanal tuval (beyaz arka planlı numpy dizisi) üzerinde tut
5. Kamera görüntüsü ile tuvali `cv.addWeighted()` ile şeffaf üst üste getir
6. Ek özellikler:
   - `c` ile tuvali temizle
   - `1/2/3/4` ile çizgi kalınlığını değiştir
   - `s` ile tuvali PNG olarak kaydet
   - Farklı renkte nesnelerle farklı renkte çizgi

### Düşünmeniz Gereken Sorular:
- `cv.moments()` ile merkez hesaplamak neden konturun en düşük/en yüksek noktasını almaktan daha sağlam?
- Hızlı hareket sırasında kareler arası "atlayan" çizgiyi düzeltmek için hangi enterpolasyon yöntemi kullanılabilir?
- Bu proje Microsoft'un Kinect bazlı PowerPoint kontrolü ile aynı mantığı taşıyor — fark nedir?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

tuval = np.ones((480, 640, 3), dtype=np.uint8) * 255  # beyaz tuval
onceki_merkez = None

lower_red = np.array([160, 100, 100])
upper_red = np.array([179, 255, 255])

# TODO: VideoCapture döngüsü
```

---
## ❓ Soru 4 — Gerçek Zamanlı Video Karakter Sanatı (ASCII Art)

**Zorluk: ⭐⭐⭐☆☆**

Kameradan gelen görüntüyü ASCII karakterleri kullanarak hem terminal hem de OpenCV penceresi üzerinde gerçek zamanlı gösteren bir program yazın.

### Görev:
1. Her kareyi küçük blok boyutuna (örn. 8×8 piksel) böl
2. Her bloğun ortalama parlaklığını hesapla
3. Parlaklığı bir ASCII karakterine eşle: `" .:-=+*#%@"` (açıktan koyuya)
4. **Versiyon A — Terminal:** `print()` ile ASCII karakterleri yazdır
5. **Versiyon B — OpenCV:** `cv.putText()` ile boş siyah görüntüye karakterleri yaz
6. `c` tuşuyla renkli/gri geçişi: gri → karakter rengi sabit, renkli → karakterin rengi o bloğun ortalama BGR rengi

### Düşünmeniz Gereken Sorular:
- ASCII karakter seti 10 seviye yeterli mi? Daha fazlası kaliteyi artırır mı?
- Terminal ve OpenCV çıktısının görsel kalitesi neden farklı?
- Bu teknik, 1970'lerde ekranda görüntü olmadan görsel içerik paylaşmada nasıl kullanılıyordu?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

ASCII_KARAKTERLER = " .:-=+*#%@"

def kareyi_ascii_donustur(frame, blok_boyutu=8):
    """
    frame'i ASCII karakter matrisine çevirir.
    Döndürür: 2D string listesi (her eleman bir karakter)
    """
    gri = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    h, w = gri.shape
    # TODO: blok blok parlaklık hesapla, karakter eşle
    pass

video = cv.VideoCapture(0)
# TODO: döngü
```

---
## ❓ Soru 5 — Çoklu Kamera Mozaik Görüntüleyici

**Zorluk: ⭐⭐⭐⭐⭐**

Birden fazla kameradan (veya video dosyasından) gelen görüntüleri tek bir mozaik pencerede gerçek zamanlı gösteren, tıklanan kamerayı tam ekrana açabilen bir güvenlik kamerası arayüzü tasarlayın.

### Görev:
1. `n` kadar kaynak aç: kameralar, video dosyaları veya RTSP stream URL'leri karışık olabilir
2. Her kaynaktan kare oku, tümünü aynı boyuta resize et
3. `np.hstack` / `np.vstack` / `np.block` ile mozaik oluştur:
   - 2 kaynak → 1×2 mozaik
   - 4 kaynak → 2×2 mozaik
   - 9 kaynak → 3×3 mozaik
4. Mouse callback: tıklanan kameranın indeksini hesapla, tam ekran göster
5. Her kamera görüntüsünün köşesine kaynak adı ve anlık FPS yaz
6. Bir kaynak açılamıyorsa (kamera bağlı değil) o hücreyi "Kaynak Yok" yazan kırmızı dikdörtgenle doldur

### Düşünmeniz Gereken Sorular:
- Senkron okuma (bir döngüde tüm kameralar sırayla) ile asenkron okuma (her kamera ayrı thread) arasındaki tradeoff nedir?
- Python'da threading ve GIL (Global Interpreter Lock) video okuma performansını nasıl etkiler?
- Profesyonel NVR (Network Video Recorder) sistemleri mozaik oluştururken hangi teknikleri kullanır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np
import threading
from typing import List, Union

class KameraKaynagi:
    """Thread-safe kamera okuma sınıfı."""
    def __init__(self, kaynak: Union[int, str], isim: str):
        self.isim = isim
        self.cap = cv.VideoCapture(kaynak)
        self.son_kare = None
        self.calisiyor = True
        # TODO: arka plan thread'i başlat

    def oku(self) -> np.ndarray:
        """En güncel kareyi döndür."""
        pass


class MozaikGoruntuleyici:
    def __init__(self, kaynaklar: List[KameraKaynagi], hucre_boyutu=(320, 240)):
        self.kaynaklar = kaynaklar
        self.hucre_boyutu = hucre_boyutu
        self.tam_ekran_indeks = None
        # TODO

    def mozaik_olustur(self) -> np.ndarray:
        """Tüm kameraları tek görüntüde birleştir."""
        pass
```

---
<div style="background: linear-gradient(135deg, #050510 0%, #0f0f2d 40%, #1a0a2e 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #a855f744;"><h2 style="color: #a855f7; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 6. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;">✅ <b>VideoCapture:</b> Kamera ID, video dosyası, RTSP stream; isOpened() kontrolü<br>✅ <b>video.release():</b> Zorunlu temizlik — kamera exclusive kaynak<br>✅ <b>Renk kanalı swap:</b> Önce .copy() al, sonra değiştir — referans tuzağı!<br>✅ <b>VideoWriter + FourCC:</b> XVID, MJPG, mp4v codec seçimi ve kayıt<br>✅ <b>Piksel okuma/yazma:</b> resim[y,x] ve resim[y1:y2, x1:x2] dilimleme<br>✅ <b>ROI (Region of Interest):</b> NumPy dilimleme = verimli kırpma<br>✅ <b>Mouse Callback:</b> EVENT_LBUTTONDOWN/UP/MOVE, setMouseCallback()<br>✅ <b>State Machine:</b> Dondur/devam döngüsü — GUI, oyun, robotik temeli<br>✅ <b>Renk uzayları:</b> BGR→HSV, BGR→LAB, BGR→BGRA — cvtColor()<br>✅ <b>Zaman bükme filtresi:</b> FIFO tamponu ile gecikme efekti<br>✅ <b>frame.copy() önemi:</b> View vs kopya, çoklu işlem için şart</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">7. Hafta → HSV Renk Tespiti, Maskeleme, Eşikleme & Adaptif Algoritmalar 🎭</p></div>